# Logística Predictiva: Optimizando la Entrega en E-commerce para Proteger la Reputación de Marca

**Autora:** Irene  
**Bootcamp:** Data Science  
**Fecha:** Mayo 2026  

---

## I. Introducción

### Contexto del problema

Una empresa internacional de comercio electrónico especializada en productos tecnológicos enfrenta un problema crítico: el **59.7% de sus envíos llegan tarde**. En un sector donde la experiencia de entrega es uno de los principales factores de fidelización, esta cifra representa un riesgo directo para la reputación de marca y la satisfacción del cliente.

El problema no es solo operativo. Cuando un cliente recibe su pedido tarde sin haber recibido ninguna comunicación previa, la experiencia negativa se amplifica: el cliente reclama enfadado, deja una reseña negativa y, en muchos casos, no vuelve a comprar. Sin embargo, si la empresa hubiera detectado ese retraso de antemano y hubiera enviado un cupón de descuento preventivo, la misma situación se convierte en una oportunidad de fidelización.

**La pregunta central del proyecto es:** ¿podemos predecir si un paquete llegará tarde antes de que salga del almacén, para que el equipo de Atención al Cliente pueda actuar de forma proactiva?

### Objetivos y alcance

- Construir un **modelo supervisado** que prediga si un envío llegará tarde (clasificación binaria)
- Desarrollar un **modelo no supervisado** (clustering) que identifique perfiles de envío con distinto nivel de riesgo
- Traducir los resultados a **acciones concretas de negocio** con impacto medible
- Demostrar que el problema puede resolverse con Machine Learning aplicado a datos logísticos reales

---

## II. Dataset

### Descripción

| Campo | Detalle |
|---|---|
| **Origen** | Kaggle — E-Commerce Shipping Dataset |
| **Observaciones** | 10.999 registros |
| **Variables** | 12 (7 numéricas, 4 categóricas, 1 target) |
| **Valores nulos** | Ninguno |
| **Duplicados** | Ninguno |
| **Target** | `Reached.on.Time_Y.N` (1=retraso, 0=a tiempo) |

### Variables del dataset

| Variable | Tipo | Descripción |
|---|---|---|
| `Warehouse_block` | Categórica | Bloque del almacén (A, B, C, D, F) |
| `Mode_of_Shipment` | Categórica | Modo de envío (Barco, Avión, Carretera) |
| `Customer_care_calls` | Numérica | Llamadas al servicio de atención al cliente |
| `Customer_rating` | Numérica | Valoración del cliente (1=peor, 5=mejor) |
| `Cost_of_the_Product` | Numérica | Coste del producto en USD |
| `Prior_purchases` | Numérica | Número de compras anteriores del cliente |
| `Product_importance` | Categórica | Importancia del producto (low, medium, high) |
| `Gender` | Categórica | Género del cliente |
| `Discount_offered` | Numérica | Descuento aplicado al producto (%) |
| `Weight_in_gms` | Numérica | Peso del paquete en gramos |
| `Reached.on.Time_Y.N` | **Target** | 1=retraso, 0=a tiempo |

### Análisis Exploratorio (EDA)

#### Distribución del target

El dataset presenta un desequilibrio moderado: el **59.7% de los envíos llegan tarde**. Esta cifra valida la urgencia del proyecto y condiciona la elección de métricas (el Accuracy por sí solo no es suficiente).

#### Hallazgo clave: el efecto del descuento

El descuento ofrecido es el predictor más potente del retraso. Con un descuento superior al 10.5%, la probabilidad de retraso se dispara al **100%**: no existe ningún envío con descuento alto que llegue a tiempo en el dataset.

![Distribución de descuentos por estado de entrega](resources/img/18_distribucion_descuentos.png)

#### Correlación entre variables

La correlación más alta con el target es `Discount_offered` (0.40). El peso tiene una correlación negativa (-0.27), lo que sugiere que los paquetes más pesados tienden a llegar mejor, un resultado contraintuitivo que el clustering confirmará más adelante.

---

## III. Preprocesamiento de los datos

### Verificación de la calidad

- **Valores nulos:** ninguno en ninguna variable
- **Duplicados:** ninguna fila duplicada
- **Outliers:** los descuentos altos (>10%) se identificaron como valores atípicos para la clase puntual, pero se mantuvieron porque tienen altísimo valor predictivo. Eliminarlos habría destruido la señal más importante del dataset.

### Decisiones de transformación

| Paso | Variable(s) | Decisión | Motivo |
|---|---|---|---|
| Eliminación | `ID` | Drop | Identificador sin valor predictivo |
| Ordinal Encoding | `Product_importance` | low=1, medium=2, high=3 | Preserva la jerarquía lógica |
| One-Hot Encoding | `Warehouse_block`, `Mode_of_Shipment`, `Gender` | get_dummies con drop_first=True | Sin orden natural, evita multicolinealidad |
| Conversión de tipos | Columnas booleanas | astype(int) | Compatibilidad con sklearn |
| Escalado | Todas las variables | StandardScaler | Necesario para Regresión Logística y KNN |

### División train/test

- **80% entrenamiento:** 8.799 muestras
- **20% test:** 2.200 muestras
- **stratify=y:** garantiza la misma proporción de retrasos (59.7%) en ambos conjuntos
- **random_state=42:** resultados reproducibles

---

## IV. Modelado

### Métrica principal: Recall

En este proyecto no usamos el Accuracy como métrica principal. La razón es el caso de negocio: **un retraso no detectado es un cliente que reclama sin haber recibido ningún cupón preventivo**. Ese es el error más costoso.

El **Recall** mide exactamente eso: de todos los retrasos reales, ¿cuántos detectó el modelo? Lo complementamos con el **ROC-AUC** (capacidad discriminativa global) y el **F1-Score** (equilibrio entre Precision y Recall).

### IV.a Modelos supervisados

Se entrenaron y compararon **5 modelos** representando 5 familias distintas de algoritmos:

| Modelo | Familia |
|---|---|
| Regresión Logística | Modelos lineales |
| Árbol de Decisión | Reglas interpretables |
| Random Forest | Ensamble en paralelo (bagging) |
| XGBoost | Ensamble en serie (boosting) |
| KNN | Similitud entre casos |

#### Resultados comparativos

![Comparativa de los 5 modelos — todas las métricas](resources/img/08_comparativa_5_modelos.png)

![Curvas ROC — comparativa de los 5 modelos](resources/img/09_curva_roc_comparativa_5_modelos.png)

| Modelo | Accuracy | Precision | Recall | F1-Score | ROC-AUC |
|---|---|---|---|---|---|
| Regresión Logística | 64.0% | 70.9% | **67.4%** | **69.1%** | 0.717 |
| Árbol de Decisión | 68.0% | 96.9% | 47.8% | 64.0% | 0.736 |
| **Random Forest** | **65.9%** | **76.4%** | **61.9%** | **68.4%** | **0.735** |
| XGBoost | 67.5% | 97.2% | 47.0% | 63.3% | **0.748** |
| KNN | 62.3% | 69.6% | 65.4% | 67.5% | 0.691 |

#### Análisis de Falsos Negativos

El Falso Negativo es el error más costoso: un retraso que el modelo no detectó y que llegó al cliente sin atención proactiva.

| Modelo | Falsos Negativos |
|---|---|
| XGBoost | 696 ❌ |
| Árbol de Decisión | 685 ❌ |
| Random Forest | 500 ✅ |
| KNN | 454 ✅ |
| Regresión Logística | 428 ✅ |

### IV.b Selección del modelo final

**Modelo elegido: Random Forest**

La Regresión Logística tiene el mejor Recall y F1, pero su ROC-AUC (0.717) es el segundo más bajo, lo que limita su potencial de mejora futura. El Random Forest combina un Recall sólido (61.9%), el segundo mejor ROC-AUC (0.735), un F1 competitivo (68.4%) y además proporciona el ranking de importancia de variables, una herramienta de valor directo para el negocio.

XGBoost y el Árbol de Decisión quedan descartados por su Recall demasiado bajo (47%), que dejaría sin detectar prácticamente la mitad de los retrasos.

#### Matriz de confusión — Random Forest

![Matriz de confusión Random Forest](resources/img/04_matriz_confusion_random_forest.png)

#### Importancia de variables — Random Forest

![Importancia de variables Random Forest](resources/img/05_importancia_variables_random_forest.png)

Las 3 variables más importantes concentran el **68% del poder predictivo**:

1. `Weight_in_gms` (28%) — el peso es el factor más determinante
2. `Discount_offered` (23%) — confirmado por el EDA
3. `Cost_of_the_Product` (17%) — productos caros implican mayor riesgo

### IV.c Modelo no supervisado — K-Means

Se aplicó K-Means sobre las 7 variables numéricas originales (sin el target) para descubrir si existen perfiles naturales de envío con distinto nivel de riesgo.

#### Elección del K óptimo

![Método del codo](resources/img/11_metodo_codo_inercia.png)

![Índice de Silhouette por K](resources/img/12_indice_silhouette_por_k.png)

![Diagrama de Silhouette comparativa K=2,3,4,5](resources/img/13_diagrama_silhouette_comparativa_k2_k3_k4_k5.png)

El Método del Codo no muestra un punto de inflexión dramático (habitual en datos de negocio reales), pero el Índice de Silhouette tiene su pico máximo en **K=3 (0.2386)**, cayendo notablemente en K=4 (0.193). El diagrama de Silhouette confirma que K=3 es el único K donde los tres bloques son simultáneamente equilibrados y mayoritariamente por encima de la media global.

#### Resultados del clustering

![Resultados del clustering K-Means K=3](resources/img/15_resultados_clustering_kmeans_k3.png)

![Perfil de variables por cluster](resources/img/16_perfil_variables_por_cluster.png)

![Visualización clusters PCA](resources/img/17_visualizacion_clusters_pca.png)

| Cluster | Tamaño | Tasa retraso | Característica definitoria |
|---|---|---|---|
| 🟣 Cluster 0 | 2.294 (20.9%) | **99.5%** | Descuento alto (media 40.1%) |
| 🟢 Cluster 1 | 6.097 (55.4%) | 47.9% | Paquete pesado (media 4.801g) |
| 🔵 Cluster 2 | 2.608 (23.7%) | 52.3% | Cliente fidelizado (5.1 compras previas) |

---

## V. Predicción y resultados finales

### Solución final

El sistema de alerta proactiva combina dos modelos complementarios:

**Random Forest** (supervisado) → predice si un envío concreto llegará tarde antes de que salga del almacén. Si la predicción es retraso, Atención al Cliente envía un cupón preventivo al cliente.

**K-Means** (no supervisado) → identifica a qué perfil de riesgo pertenece cada envío. Permite actuar a nivel estratégico (segmentos) además de a nivel operativo (envío individual).

### Impacto en negocio

Sobre los **2.200 envíos del conjunto de test**:

| Resultado | Número | Significado |
|---|---|---|
| Retrasos detectados correctamente | **813** | Clientes que reciben el cupón preventivo |
| Retrasos no detectados | 500 | Clientes que reclaman sin atención previa |
| Falsas alarmas | 251 | Cupones enviados innecesariamente |
| Envíos puntuales correctos | 636 | Sin intervención, llegan a tiempo |

El modelo detecta **6 de cada 10 retrasos** antes de que ocurran. Un cupón enviado de más es un coste asumible. Un cliente que reclama sin atención previa es un daño de reputación evitable.

### El hallazgo más importante del proyecto

El descuento alto como factor crítico aparece confirmado por **tres metodologías independientes**:

1. **EDA:** con descuento > 10.5%, el 100% de los envíos llegan tarde (observación estadística)
2. **Árbol de Decisión:** el descuento es la primera pregunta del árbol, con pureza absoluta (gini=0) en la rama de descuento alto (regla supervisada)
3. **K-Means:** sin ver ninguna etiqueta de retraso, agrupa espontáneamente los envíos con descuento alto en un cluster con tasa de retraso del 99.5% (confirmación no supervisada)

Tres análisis distintos, la misma conclusión. Es la señal más robusta y fiable del proyecto.

---

## VI. Conclusiones y futuros pasos

### Conclusiones

**Lo que funciona:**
- El modelo Random Forest detecta el 61.9% de los retrasos antes de que ocurran, con una Precision del 76.4%. Es un resultado sólido y accionable para un sistema de alertas proactivas.
- El clustering K-Means identificó un segmento de altísimo riesgo (Cluster 0: 99.5% de retraso) que la empresa puede gestionar de forma sistemática con una regla simple: descuento > 25% → alerta automática.
- La coincidencia de los tres análisis (EDA, árbol, clustering) sobre el descuento como factor crítico da robustez y credibilidad a las conclusiones.

**Limitaciones:**
- El Recall del 61.9% significa que 4 de cada 10 retrasos siguen sin detectarse. Hay margen de mejora significativo.
- El dataset no incluye variables externas (tráfico, clima, temporada) que podrían mejorar considerablemente el modelo.
- Los valores moderados del Índice de Silhouette en el clustering (máx. 0.24) indican que los grupos no están perfectamente separados en el espacio multidimensional.

### Futuros pasos

| Mejora | Impacto esperado |
|---|---|
| GridSearchCV para optimizar hiperparámetros del Random Forest | Mejora del Recall y ROC-AUC |
| Bajar el umbral de decisión de XGBoost del 50% al 25-30% | XGBoost podría superar al Random Forest en Recall |
| Añadir la variable Cluster como feature al modelo supervisado | Potencial mejora del Recall del Random Forest |
| Explorar DBSCAN como alternativa al K-Means | No requiere definir K y detecta outliers como cluster propio |
| Enriquecer el dataset con datos de tráfico, clima o temporada | Mayor capacidad predictiva del modelo |
| Demo en Streamlit para Atención al Cliente | Impacto directo en el negocio |

---

### Estructura del repositorio

```
ML_SHIPPING/
└── src/
    ├── data/
    │   ├── shipping_data.csv
    │   ├── train.csv
    │   └── test.csv
    ├── model/
    │   ├── production/
    │   │   └── random_forest_final.pkl
    │   ├── arbol_decision.pkl
    │   ├── knn.pkl
    │   ├── random_forest.pkl
    │   ├── regresion_logistica.pkl
    │   └── xgboost.pkl
    ├── notebooks/
    │   ├── 01_exploración_y_eda.ipynb
    │   ├── 02_preprocesamiento_y_modelado.ipynb
    │   └── 03_clustering_kmeans.ipynb
    ├── resources/
    │   └── img/
    ├── utils/
    │   ├── __init__.py
    │   ├── metricas.py
    │   └── preprocesamiento.py
    ├── memoria.ipynb
    └── README.md
```